# LeetCode 293 — Flip Game bằng Python

**Link đề:** https://leetcode.com/problems/flip-game/
**Độ khó:** Easy · **Chủ đề:** String · **Bài Premium**

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
from typing import List

class Solution:
    def generatePossibleNextMoves(self, currentState: str) -> List[str]:
        ...
```

## Mục tiêu học
1. Hiểu **chuỗi (str) là bất biến** (immutable) trong Python và cách "sửa" chuỗi bằng **slicing** để tạo chuỗi mới.
2. Duyệt **cặp ký tự kề nhau** `s[i], s[i+1]` an toàn với `range(len(s) - 1)`.
3. Viết 4 cách: ghép slice, đổi sang `list` rồi sửa và hoàn tác, dùng `str.find` nhảy tới vị trí tiếp theo, và list comprehension.
4. Trace từng vị trí `i`, phân tích Time/Space khi **kết quả** cũng tốn bộ nhớ.
5. Chuẩn bị cho bài tiếp theo **Flip Game II (294)**: dùng các nước đi sinh ra để tìm kiếm thắng/thua.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — liệt kê mọi trạng thái sau một nước đi

Bạn chơi Flip Game với một người bạn. Cho chuỗi `currentState` chỉ gồm ký tự `'+'` và `'-'`.

- Mỗi lượt, người chơi chọn **hai ký tự `'+'` liên tiếp** (`"++"`) và lật chúng thành `"--"`.
- Ai không còn nước đi thì thua (phần này để dành cho bài 294).

Yêu cầu bài 293: trả về **tất cả** trạng thái có thể có của `currentState` **sau đúng một nước đi hợp lệ**. Kết quả trả về theo **thứ tự bất kỳ**. Nếu không có nước đi nào, trả về `[]`.

| Input `currentState` | Output | Giải thích |
|---|---|---|
| `"++++"` | `["--++", "+--+", "++--"]` | có 3 cặp `"++"` ở vị trí 0, 1, 2 |
| `"+"` | `[]` | không đủ 2 ký tự |
| `"+-+-"` | `[]` | không có hai dấu `+` kề nhau |

**Constraints (theo đề):** `1 <= currentState.length <= 500`; `currentState[i]` là `'+'` hoặc `'-'`.

Câu hỏi đặt ra trước khi code:
- Chuỗi Python có sửa trực tiếp `s[i] = '-'` được không?
- Duyệt tới đâu để đọc `s[i+1]` mà không vượt chỉ số?
- Các cặp `"++"` chồng lên nhau (như trong `"+++"`) có tính riêng không?

In [1]:
s = "++++"
# Liệt kê các vị trí i có cặp "++" bắt đầu tại i
print([i for i in range(len(s) - 1) if s[i] == '+' and s[i + 1] == '+'])   # [0, 1, 2]

[0, 1, 2]


## 2. Nền tảng Python cho bài này

### 2.1. Indexing chuỗi
`s[i]` lấy ký tự thứ `i` (đếm từ 0). `len(s)` là độ dài. Chỉ số hợp lệ: `0 .. len(s) - 1`.

In [2]:
s = "+-++"
print(len(s))        # 4
print(s[0], s[1])    # + -
print(s[-1])         # +  (chỉ số âm: đếm từ cuối)
try:
    print(s[4])
except IndexError as e:
    print("IndexError:", e)

4
+ -
+
IndexError: string index out of range


### 2.2. Chuỗi là bất biến (immutable)

Ta **không thể** gán `s[0] = '-'`. Muốn "sửa", ta phải **tạo chuỗi mới**.

In [3]:
s = "++++"
try:
    s[0] = '-'
except TypeError as e:
    print("TypeError:", e)

TypeError: 'str' object does not support item assignment


### 2.3. Slicing `s[a:b]` để cắt và ghép

- `s[:i]` là phần **trước** vị trí `i` (không gồm `i`).
- `s[i+2:]` là phần **từ** vị trí `i+2` tới hết.
- Ghép `s[:i] + "--" + s[i+2:]` = thay 2 ký tự tại `i, i+1` bằng `"--"`.

Ví dụ `s = "++++"`, `i = 1`: `s[:1] = "+"`, `s[3:] = "+"` → `"+" + "--" + "+" = "+--+"`.

In [4]:
s = "++++"
i = 1
print(repr(s[:i]), repr(s[i + 2:]))
print(s[:i] + "--" + s[i + 2:])   # +--+
print(repr(s[4:]))                # slice vượt cuối KHÔNG lỗi, chỉ ra chuỗi rỗng ''

'+' '+'
+--+
''


### 2.4. `range(len(s) - 1)` — duyệt cặp kề nhau

Vì ta đọc `s[i + 1]`, chỉ số `i` lớn nhất được phép là `len(s) - 2`. `range(len(s) - 1)` sinh `0 .. len(s) - 2`.

Khi `len(s) == 1`, `range(0)` rỗng → vòng lặp không chạy → tự động trả `[]`. Không cần `if` riêng.

In [5]:
for s in ["++++", "+", "+-"]:
    pairs = [(i, s[i], s[i + 1]) for i in range(len(s) - 1)]
    print(repr(s), "->", pairs)

'++++' -> [(0, '+', '+'), (1, '+', '+'), (2, '+', '+')]
'+' -> []
'+-' -> [(0, '+', '-')]


### 2.5. So sánh một đoạn con: `s[i:i+2] == "++"`

Thay vì hai điều kiện `s[i] == '+' and s[i+1] == '+'`, ta có thể cắt đoạn dài 2 rồi so sánh. Cả hai cách cho cùng kết quả.

### 2.6. `list(s)` và `"".join(chars)`
- `list("+-+")` → `['+', '-', '+']`: list **sửa được**.
- `"".join(['-', '-', '+'])` → `"--+"`: ghép list ký tự thành chuỗi.

In [6]:
s = "+++-"
print([s[i:i + 2] for i in range(len(s) - 1)])   # ['++', '++', '+-']
chars = list(s)
chars[0] = chars[1] = '-'
print(chars, "->", "".join(chars))

['++', '++', '+-']
['-', '-', '+', '-'] -> --+-


## 3. Cách trực giác — với mỗi vị trí, kiểm tra và tạo chuỗi mới

Ý tưởng tự nhiên nhất:
1. Duyệt mọi `i` từ `0` tới `len(s) - 2`.
2. Nếu `s[i]` và `s[i+1]` đều là `'+'`: tạo chuỗi mới bằng cách thay hai ký tự đó bằng `"--"` và thêm vào kết quả.

Đây đã là lời giải tối ưu về Big-O (vì kết quả bắt buộc chứa tới `n - 1` chuỗi dài `n`), nên bài này không có "brute force chậm" thật sự. Những cách khác dưới đây giúp luyện cú pháp và chuẩn bị cho biến thể.

In [7]:
def flip_slice(s: str) -> list[str]:
    result = []
    for i in range(len(s) - 1):
        if s[i] == '+' and s[i + 1] == '+':
            result.append(s[:i] + "--" + s[i + 2:])
    return result

print(flip_slice("++++"))
print(flip_slice("+"))
print(flip_slice("+-+-"))

['--++', '+--+', '++--']
[]
[]


## 4. "Nút thắt" thực sự của bài: hiểu chi phí và tránh lỗi

Bài không khó về thuật toán, mà dễ sai ở các điểm sau:
- **Sửa trực tiếp chuỗi** → `TypeError`.
- **Dùng chung một list** cho mọi nước đi mà quên hoàn tác → các kết quả sau bị "dính" dấu `-` của nước trước.
- **Vượt chỉ số** khi đọc `s[i+1]` với `i = len(s) - 1`.
- **Bỏ sót cặp chồng nhau**: `"+++"` có **2** nước đi (vị trí 0 và 1), không phải 1.

Về chi phí: mỗi chuỗi mới tốn `O(n)` để tạo. Có tối đa `n - 1` chuỗi → tổng `O(n²)`. Với `n <= 500` thì tối đa ~250.000 ký tự — rất nhỏ.

In [8]:
print(flip_slice("+++"))              # 2 nước đi chồng nhau: ['--+', '+--']
print(len(flip_slice("+" * 500)))     # 499 chuỗi
print(sum(len(t) for t in flip_slice("+" * 500)))   # 499 * 500 ký tự

['--+', '+--']
499
249500


## 5. Insight — mỗi nước đi tương ứng đúng một vị trí `i` có `"++"`

Ánh xạ một-một:

```text
nước đi hợp lệ  <->  chỉ số i sao cho s[i:i+2] == "++"
```

- Hai `i` khác nhau cho hai chuỗi kết quả khác nhau (vì vị trí bị lật khác nhau). Nên **không có trùng lặp**, không cần `set`.
- Mọi nước đi hợp lệ đều có dạng này. Nên duyệt đủ mọi `i` là **không bỏ sót**.

Từ insight này có thêm cách dùng `str.find("++", start)` để **nhảy thẳng** tới cặp tiếp theo thay vì kiểm tra từng ký tự.

In [9]:
s = "+-++-+++"
print(s.find("++"))        # 2: vị trí đầu tiên có "++"
print(s.find("++", 3))     # 5: tìm tiếp từ vị trí 3
print(s.find("++", 6))     # 6: "++" ở vị trí 6-7 (chồng với cặp 5-6)
print(s.find("++", 7))     # -1: không còn

2
5
6
-1


## 6. Lời giải chính — giảng từng đoạn rồi ghép

**Đoạn 1 — chuẩn bị kết quả:**
```python
result = []
```

**Đoạn 2 — duyệt cặp kề nhau an toàn:**
```python
for i in range(len(currentState) - 1):
```

**Đoạn 3 — điều kiện nước đi hợp lệ:**
```python
if currentState[i:i + 2] == "++":
```

**Đoạn 4 — tạo chuỗi mới, không sửa chuỗi cũ:**
```python
result.append(currentState[:i] + "--" + currentState[i + 2:])
```

**Đoạn 5 — trả kết quả** (`[]` nếu không có nước nào).

In [10]:
def generate_moves(current_state: str) -> list[str]:
    result = []
    for i in range(len(current_state) - 1):
        if current_state[i:i + 2] == "++":
            result.append(current_state[:i] + "--" + current_state[i + 2:])
    return result

for s in ["++++", "+", "--", "++", "+-++-+++"]:
    print(repr(s), "->", generate_moves(s))

'++++' -> ['--++', '+--+', '++--']
'+' -> []
'--' -> []
'++' -> ['--']
'+-++-+++' -> ['+----+++', '+-++---+', '+-++-+--']


## 7. TRACE từng bước

Bảng tay cho `s = "+-++-+++"` (độ dài 8, `i` chạy 0..6):

| i | `s[i:i+2]` | Hợp lệ? | Chuỗi mới |
|---:|:---:|:---:|---|
| 0 | `+-` | không | — |
| 1 | `-+` | không | — |
| 2 | `++` | **có** | `+----+++` |
| 3 | `+-` | không | — |
| 4 | `-+` | không | — |
| 5 | `++` | **có** | `+-++---+` |
| 6 | `++` | **có** | `+-++-+--` |

Cách tính dòng 5: `s[:5] = "+-++-"`, `s[7:] = "+"` → `"+-++-" + "--" + "+" = "+-++---+"`. Chú ý: chỉ 2 ký tự tại 5, 6 bị lật; ký tự thứ 7 giữ nguyên.
Dòng 6: `s[:6] = "+-++-+"`, `s[8:] = ""` (rỗng vì vượt cuối) → `"+-++-+--"`.

In [11]:
def trace_generate_moves(s: str) -> list[str]:
    result = []
    print(f"s = {s!r}, len = {len(s)}, i chạy 0..{len(s) - 2}")
    for i in range(len(s) - 1):
        pair = s[i:i + 2]
        if pair == "++":
            new_state = s[:i] + "--" + s[i + 2:]
            result.append(new_state)
            print(f"i={i}: pair={pair!r} -> HỢP LỆ | {s[:i]!r} + '--' + {s[i + 2:]!r} = {new_state!r}")
        else:
            print(f"i={i}: pair={pair!r} -> bỏ qua")
    print("Kết quả:", result)
    return result

trace_generate_moves("+-++-+++")

s = '+-++-+++', len = 8, i chạy 0..6
i=0: pair='+-' -> bỏ qua
i=1: pair='-+' -> bỏ qua
i=2: pair='++' -> HỢP LỆ | '+-' + '--' + '-+++' = '+----+++'
i=3: pair='+-' -> bỏ qua
i=4: pair='-+' -> bỏ qua
i=5: pair='++' -> HỢP LỆ | '+-++-' + '--' + '+' = '+-++---+'
i=6: pair='++' -> HỢP LỆ | '+-++-+' + '--' + '' = '+-++-+--'
Kết quả: ['+----+++', '+-++---+', '+-++-+--']


['+----+++', '+-++---+', '+-++-+--']

Đối chiếu trace với bảng tay: cặp ở `i = 5` và `i = 6` **chồng nhau** (dùng chung ký tự thứ 6) nhưng vẫn là hai nước đi khác nhau. Lỗi hay gặp khi tính tay là thay luôn cả phần đuôi (`+-++----`) — trace bằng code giúp bắt lỗi đó.

### 7.1. Trace ví dụ biên

In [12]:
trace_generate_moves("+")
print()
trace_generate_moves("++")

s = '+', len = 1, i chạy 0..-1
Kết quả: []

s = '++', len = 2, i chạy 0..0
i=0: pair='++' -> HỢP LỆ | '' + '--' + '' = '--'
Kết quả: ['--']


['--']

## 8. Các lời giải khác (luyện cú pháp)

### Cách 2 — đổi sang list, sửa rồi **hoàn tác** (backtracking mini)

Ý tưởng dùng lại đúng một list `chars`: lật → ghi nhận → lật lại. Đây là khuôn mẫu của **backtracking** mà bài 294 sẽ cần.

In [13]:
def flip_list_undo(s: str) -> list[str]:
    chars = list(s)
    result = []
    for i in range(len(chars) - 1):
        if chars[i] == '+' and chars[i + 1] == '+':
            chars[i] = chars[i + 1] = '-'      # thực hiện nước đi
            result.append("".join(chars))      # ghi nhận trạng thái
            chars[i] = chars[i + 1] = '+'      # hoàn tác (undo)
    return result

print(flip_list_undo("++++"))

['--++', '+--+', '++--']


### Cách 3 — `str.find` nhảy tới cặp tiếp theo

`find` trả `-1` khi không còn. Sau khi gặp cặp ở `i`, tìm tiếp từ `i + 1` (không phải `i + 2`) để **không bỏ sót cặp chồng nhau**.

In [14]:
def flip_find(s: str) -> list[str]:
    result = []
    i = s.find("++")
    while i != -1:
        result.append(s[:i] + "--" + s[i + 2:])
        i = s.find("++", i + 1)
    return result

print(flip_find("++++"))
print(flip_find("+-++-+++"))

['--++', '+--+', '++--']
['+----+++', '+-++---+', '+-++-+--']


### Cách 4 — list comprehension một dòng

**List comprehension** `[biểu_thức for x in dãy if điều_kiện]` gói vòng `for` + `if` + `append` vào một biểu thức.

In [15]:
def flip_comprehension(s: str) -> list[str]:
    return [s[:i] + "--" + s[i + 2:] for i in range(len(s) - 1) if s[i:i + 2] == "++"]

print(flip_comprehension("++++"))

['--++', '+--+', '++--']


## 9. Code nộp LeetCode

### Cách chính — duyệt và ghép slice

In [16]:
from typing import List

class Solution:
    def generatePossibleNextMoves(self, currentState: str) -> List[str]:
        result = []
        for i in range(len(currentState) - 1):
            if currentState[i] == '+' and currentState[i + 1] == '+':
                result.append(currentState[:i] + "--" + currentState[i + 2:])
        return result

print(Solution().generatePossibleNextMoves("++++"))
print(Solution().generatePossibleNextMoves("+"))

['--++', '+--+', '++--']
[]


### Cách nộp thay thế — list comprehension

In [17]:
class SolutionComprehension:
    def generatePossibleNextMoves(self, currentState: str) -> List[str]:
        s = currentState
        return [s[:i] + "--" + s[i + 2:] for i in range(len(s) - 1) if s[i:i + 2] == "++"]

print(SolutionComprehension().generatePossibleNextMoves("++++"))

['--++', '+--+', '++--']


## 10. Vì sao đúng?

**Invariant của vòng lặp:** sau khi xử lý xong chỉ số `i`, `result` chứa **đúng** các trạng thái sinh ra từ các cặp `"++"` bắt đầu tại `0..i`, theo thứ tự tăng dần của vị trí.

- **Không bỏ sót:** mọi nước đi hợp lệ lật 2 ký tự kề nhau ở vị trí `j, j+1` với `0 <= j <= n-2` và cả hai là `'+'`; vòng lặp đi qua mọi `j` như vậy.
- **Không thừa:** chỉ thêm khi điều kiện `"++"` đúng.
- **Không trùng:** xét hai vị trí `j < k`. Trong chuỗi sinh từ `j`, ký tự tại `j` là `'-'`. Trong chuỗi sinh từ `k`, chỉ các vị trí `k, k+1` (đều > `j`) bị đổi, nên ký tự tại `j` vẫn là `'+'`. Hai chuỗi khác nhau tại vị trí `j`.
- **Chuỗi gốc không đổi** vì chuỗi bất biến; mỗi kết quả được tạo mới từ chuỗi gốc.

## 11. Time / Space Complexity và so sánh

Gọi `n = len(currentState)`, `m` = số cặp `"++"` (`m <= n - 1`).

| Cách | Time | Space phụ (ngoài kết quả) | Ghi chú |
|---|---|---|---|
| Ghép slice | `O(n + m·n)` = `O(n²)` worst | `O(1)` | dễ đọc nhất |
| List + hoàn tác | `O(n + m·n)` | `O(n)` cho `chars` | khuôn mẫu backtracking |
| `str.find` | `O(n + m·n)` | `O(1)` | nhảy nhanh khi ít cặp `++` |
| Comprehension | `O(n + m·n)` | `O(1)` | ngắn gọn |

**Kết quả** luôn tốn `O(m·n)` bộ nhớ (m chuỗi, mỗi chuỗi dài n). Không cách nào nhỏ hơn được, vì đề bắt trả về tất cả chuỗi.

Worst case: `"+" * n` → `m = n - 1` → `O(n²)`.

## 12. Bộ test — mẫu đề, biên, đối chiếu

Vì đề cho phép **thứ tự bất kỳ**, khi so sánh ta dùng `sorted(...)` để không phụ thuộc thứ tự.

In [18]:
def check(name, actual, expected):
    assert sorted(actual) == sorted(expected), (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề: ++++", sol.generatePossibleNextMoves("++++"), ["--++", "+--+", "++--"])
check("đề: +", sol.generatePossibleNextMoves("+"), [])
check("một ký tự -", sol.generatePossibleNextMoves("-"), [])
check("toàn -", sol.generatePossibleNextMoves("----"), [])
check("đúng 1 cặp", sol.generatePossibleNextMoves("++"), ["--"])
check("cặp chồng nhau", sol.generatePossibleNextMoves("+++"), ["--+", "+--"])
check("xen kẽ", sol.generatePossibleNextMoves("+-+-+"), [])
check("cặp ở cuối", sol.generatePossibleNextMoves("--++"), ["----"])
check("hỗn hợp", sol.generatePossibleNextMoves("+-++-+++"), ["+----+++", "+-++---+", "+-++-+--"])

PASS đề: ++++: ['--++', '+--+', '++--']
PASS đề: +: []
PASS một ký tự -: []
PASS toàn -: []
PASS đúng 1 cặp: ['--']
PASS cặp chồng nhau: ['--+', '+--']
PASS xen kẽ: []
PASS cặp ở cuối: ['----']
PASS hỗn hợp: ['+----+++', '+-++---+', '+-++-+--']


In [19]:
import random
random.seed(293)
methods = [flip_slice, flip_list_undo, flip_find, flip_comprehension,
           Solution().generatePossibleNextMoves, SolutionComprehension().generatePossibleNextMoves]
for _ in range(500):
    n = random.randint(1, 12)
    s = "".join(random.choice("+-") for _ in range(n))
    expected = flip_slice(s)
    for f in methods:
        assert f(s) == expected, (s, f)
    assert len(set(expected)) == len(expected)        # không trùng lặp
    assert all(len(t) == len(s) for t in expected)    # độ dài giữ nguyên
    assert all(t.count('+') == s.count('+') - 2 for t in expected)  # mỗi nước mất đúng 2 dấu '+'
print("500 chuỗi ngẫu nhiên: 6 cách cho cùng kết quả, các tính chất đều đúng")

500 chuỗi ngẫu nhiên: 6 cách cho cùng kết quả, các tính chất đều đúng


## 13. Lỗi người mới hay gặp

### Lỗi 1: Sửa trực tiếp chuỗi
```python
s[i] = '-'      # TypeError: 'str' object does not support item assignment
```
Sửa: tạo chuỗi mới bằng slice, hoặc đổi sang `list`.

### Lỗi 2: Dùng `list` nhưng quên hoàn tác
Kết quả sau mang theo dấu `-` của nước đi trước.

In [20]:
def flip_forgot_undo(s: str) -> list[str]:
    chars = list(s)
    result = []
    for i in range(len(chars) - 1):
        if chars[i] == '+' and chars[i + 1] == '+':
            chars[i] = chars[i + 1] = '-'
            result.append("".join(chars))
            # QUÊN: chars[i] = chars[i + 1] = '+'
    return result

print("sai :", flip_forgot_undo("++++"))   # chỉ ra ['--++', '----'] — sai số lượng lẫn nội dung
print("đúng:", flip_list_undo("++++"))

sai : ['--++', '----']
đúng: ['--++', '+--+', '++--']


### Lỗi 3: `range(len(s))` rồi đọc `s[i + 1]`
Với `i = len(s) - 1`, `s[i + 1]` gây `IndexError`. (Dùng slice `s[i:i+2]` thì không lỗi nhưng ra chuỗi dài 1, so sánh với `"++"` luôn False — chạy đúng nhưng che giấu ý đồ.)

### Lỗi 4: Nhảy `i += 2` sau khi gặp cặp
Như vậy bỏ sót cặp chồng nhau.

In [21]:
def flip_index_error(s: str) -> list[str]:
    result = []
    for i in range(len(s)):                    # SAI: tới i = len(s) - 1
        if s[i] == '+' and s[i + 1] == '+':
            result.append(s[:i] + "--" + s[i + 2:])
    return result

try:
    flip_index_error("-+")
except IndexError as e:
    print("IndexError:", e)

def flip_skip(s: str) -> list[str]:
    result, i = [], 0
    while i < len(s) - 1:
        if s[i:i + 2] == "++":
            result.append(s[:i] + "--" + s[i + 2:])
            i += 2                             # SAI: bỏ qua vị trí i + 1
        else:
            i += 1
    return result

print("sai :", flip_skip("+++"))
print("đúng:", flip_slice("+++"))

IndexError: string index out of range
sai : ['--+']
đúng: ['--+', '+--']


### Lỗi 5: Dùng `s.replace("++", "--")`
`replace` thay **mọi** lần xuất hiện (không chồng nhau) trong một chuỗi → chỉ ra **một** chuỗi, không phải danh sách nước đi.

In [22]:
print("++++".replace("++", "--"))      # '----' — lật 2 cặp cùng lúc, không phải 1 nước đi
print("++++".replace("++", "--", 1))   # '--++' — chỉ được nước đầu tiên

----
--++


## 14. Mẹo nhận dạng pattern

- Đề có **"hai ký tự liên tiếp"**, **"cặp kề nhau"** → `for i in range(len(s) - 1)` hoặc `zip(s, s[1:])`.
- Đề yêu cầu **"tất cả trạng thái sau một bước"** → đây là hàm **sinh láng giềng** (neighbors) — viên gạch cơ bản của BFS/DFS/game search.
- Khi cần "sửa" chuỗi nhiều lần: **slice + ghép** (đơn giản) hoặc **list + hoàn tác** (khi sẽ đệ quy sâu).
- Trả về "thứ tự bất kỳ" → test bằng `sorted()` hoặc `set()`.

In [23]:
s = "+-++"
for i, (a, c) in enumerate(zip(s, s[1:])):     # zip ghép từng cặp kề nhau
    print(i, a, c, "<- cặp ++" if a == c == '+' else "")

0 + - 
1 - + 
2 + + <- cặp ++


## 15. Biến thể

### Biến thể 1 — Flip Game II (LeetCode 294): người đi trước có chắc thắng không?
Dùng chính hàm sinh nước đi: `win(s) = any(not win(t) for t in moves(s))`. Thêm **memo** vì cùng một trạng thái xuất hiện qua nhiều thứ tự đi khác nhau.

In [24]:
from functools import lru_cache

@lru_cache(maxsize=None)
def can_win_flip(s: str) -> bool:
    return any(not can_win_flip(t) for t in flip_slice(s))

for s in ["++++", "+++++", "++", "+", "++++++"]:
    print(repr(s), "-> người đi trước thắng?", can_win_flip(s))

'++++' -> người đi trước thắng? True
'+++++' -> người đi trước thắng? False
'++' -> người đi trước thắng? True
'+' -> người đi trước thắng? False
'++++++' -> người đi trước thắng? True


### Biến thể 2 — chỉ đếm số nước đi
Không cần tạo chuỗi → `O(n)` thời gian, `O(1)` bộ nhớ.

### Biến thể 3 — lật `k` dấu `+` liên tiếp
Tổng quát hóa: điều kiện `s[i:i+k] == "+" * k`, thay bằng `"-" * k`, duyệt `range(len(s) - k + 1)`.

In [25]:
def count_moves(s: str) -> int:
    return sum(1 for i in range(len(s) - 1) if s[i] == s[i + 1] == '+')

def flip_k(s: str, k: int) -> list[str]:
    return [s[:i] + "-" * k + s[i + k:] for i in range(len(s) - k + 1) if s[i:i + k] == "+" * k]

print(count_moves("+-++-+++"))     # 3
print(flip_k("+++++", 3))          # ['---++', '+---+', '++---']

3
['---++', '+---+', '++---']


## 16. Bài tập tự luyện (TODO — không chạy sẵn)

### Bài 1 — Dự đoán output
Không chạy code, đoán kết quả của `generate_moves("+++-++")` và `len(generate_moves("+" * 10))`.

### Bài 2 — Tính tay slicing
Với `s = "-+++-"`, `i = 2`: `s[:i]`, `s[i+2:]` và chuỗi mới là gì?

### Bài 3 — Điền chỗ trống

In [26]:
def exercise_moves(s: str) -> list[str]:
    result = []
    for i in range(len(s) - 1):
        # TODO: nếu s[i] và s[i+1] đều là '+', thêm chuỗi đã lật vào result
        pass
    return result

# assert sorted(exercise_moves("++++")) == sorted(["--++", "+--+", "++--"])
# assert exercise_moves("+") == []

### Bài 4 — Tự viết: các vị trí của nước đi
Trả về list chỉ số `i` của mọi nước đi hợp lệ (thay vì chuỗi).

In [27]:
def exercise_move_positions(s: str) -> list[int]:
    # TODO
    pass

# assert exercise_move_positions("+-++-+++") == [2, 5, 6]
# assert exercise_move_positions("--") == []

### Bài 5 — Biến thể: lật ngược "--" thành "++"
Viết hàm sinh mọi trạng thái sau khi lật **một** cặp `"--"` thành `"++"`.

In [28]:
def exercise_reverse_flip(s: str) -> list[str]:
    # TODO
    pass

# assert sorted(exercise_reverse_flip("---")) == sorted(["++-", "-++"])
# assert exercise_reverse_flip("+-+") == []

### Bài 6 — Mini challenge: Flip Game II không dùng `lru_cache`
Viết `exercise_can_win(s)` dùng dict `memo` tự quản lý, áp dụng **list + hoàn tác** (không tạo chuỗi mới ở mỗi nhánh, chỉ `"".join` để làm key memo).

In [29]:
def exercise_can_win(s: str) -> bool:
    memo = {}
    chars = list(s)
    def dfs() -> bool:
        # TODO 1: key = "".join(chars); nếu có trong memo thì trả về
        # TODO 2: thử mọi i có chars[i] == chars[i+1] == '+': lật, gọi dfs(), hoàn tác
        # TODO 3: nếu có nước làm đối thủ thua -> True
        pass
    return dfs()

# assert exercise_can_win("++++") == True
# assert exercise_can_win("+++++") == False

## 17. Gợi ý 3 mức

**Bài 1** — (1) Liệt kê các cặp `++`. (2) `"+++-++"` có cặp ở 0, 1, 4. (3) `"+" * 10` có 9 cặp.

**Bài 2** — (1) `s[:2]` là 2 ký tự đầu. (2) `s[4:]` là từ vị trí 4. (3) Ghép `"-+" + "--" + "-"`.

**Bài 3** — (1) Cần cả hai ký tự là `+`. (2) Tạo chuỗi bằng slice, không sửa `s`. (3) `result.append(s[:i] + "--" + s[i + 2:])`.

**Bài 4** — (1) Giống bài 3 nhưng lưu `i`. (2) List comprehension. (3) `[i for i in range(len(s) - 1) if s[i:i + 2] == "++"]`.

**Bài 5** — (1) Đổi vai `+` và `-`. (2) Điều kiện `s[i:i+2] == "--"`. (3) Thay bằng `"++"`.

**Bài 6** — (1) Khuôn `win = any(not win(next))`. (2) Phải hoàn tác **trước** khi `return True`, hoặc lưu kết quả rồi hoàn tác. (3) `memo[key] = result` trước khi trả về.

## 18. Đáp án tham khảo

**Bài 1:** các cặp `++` ở vị trí 0, 1, 4 → `['--+-++', '+---++', '+++---']`; `"+" * 10` có 9 cặp → 9 nước đi.

**Bài 2:** `s[:2] = "-+"`, `s[4:] = "-"`, chuỗi mới `"-+---"`.

In [30]:
print("Bài 1:", generate_moves("+++-++"), "| số nước với '+'*10:", len(generate_moves("+" * 10)))
s = "-+++-"
print("Bài 2:", s[:2], s[4:], s[:2] + "--" + s[4:])

def answer_moves(s: str) -> list[str]:
    result = []
    for i in range(len(s) - 1):
        if s[i] == '+' and s[i + 1] == '+':
            result.append(s[:i] + "--" + s[i + 2:])
    return result

def answer_move_positions(s: str) -> list[int]:
    return [i for i in range(len(s) - 1) if s[i:i + 2] == "++"]

def answer_reverse_flip(s: str) -> list[str]:
    return [s[:i] + "++" + s[i + 2:] for i in range(len(s) - 1) if s[i:i + 2] == "--"]

def answer_can_win(s: str) -> bool:
    memo = {}
    chars = list(s)
    def dfs() -> bool:
        key = "".join(chars)
        if key in memo:
            return memo[key]
        result = False
        for i in range(len(chars) - 1):
            if chars[i] == '+' and chars[i + 1] == '+':
                chars[i] = chars[i + 1] = '-'
                opponent_wins = dfs()
                chars[i] = chars[i + 1] = '+'      # hoàn tác trước khi quyết định
                if not opponent_wins:
                    result = True
                    break
        memo[key] = result
        return result
    return dfs()

assert generate_moves("+++-++") == ["--+-++", "+---++", "+++---"]
assert len(generate_moves("+" * 10)) == 9
assert s[:2] + "--" + s[4:] == "-+---"
assert sorted(answer_moves("++++")) == sorted(["--++", "+--+", "++--"]) and answer_moves("+") == []
assert answer_move_positions("+-++-+++") == [2, 5, 6] and answer_move_positions("--") == []
assert sorted(answer_reverse_flip("---")) == sorted(["++-", "-++"]) and answer_reverse_flip("+-+") == []
assert answer_can_win("++++") is True and answer_can_win("+++++") is False
for t in ["", "+", "++", "+++", "++++++", "+++-++", "++-+++-++"]:
    assert answer_can_win(t) == can_win_flip(t), t
print("Đáp án tham khảo PASS toàn bộ test!")

Bài 1: ['--+-++', '+---++', '+++---'] | số nước với '+'*10: 9
Bài 2: -+ - -+---
Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
from typing import List

class Solution:
    def generatePossibleNextMoves(self, currentState: str) -> List[str]:
        s = currentState
        return [s[:i] + "--" + s[i + 2:] for i in range(len(s) - 1) if s[i:i + 2] == "++"]
```

**Template "sinh mọi trạng thái sau một bước":**
1. Duyệt mọi vị trí có thể đặt nước đi (`range(len(s) - k + 1)` với nước dài `k`).
2. Kiểm tra điều kiện hợp lệ.
3. Tạo trạng thái mới (slice + ghép) **hoặc** sửa → ghi nhận → hoàn tác.

**Checklist trước khi submit:**
- [ ] Không gán `s[i] = ...` trên chuỗi?
- [ ] Vòng lặp dừng ở `len(s) - 2` (dùng `range(len(s) - 1)`)?
- [ ] Không nhảy `i += 2`, giữ được cặp chồng nhau?
- [ ] Trả `[]` khi không có nước đi (kể cả chuỗi dài 1)?
- [ ] Tên method đúng `generatePossibleNextMoves`, trả về `List[str]`?